In [ ]:

import matplotlib.pyplot as plt

prompt = "The moon shines brightly"

tokens = enc.encode(prompt)

sample_input = torch.tensor([tokens]).to(DEVICE)

model(sample_input)

attention_map = (
    model.layers[0]
    .attention
    .latest_attention[0,0]
    .cpu()
    .numpy()
)

plt.figure(figsize=(8,8))

plt.imshow(attention_map)

plt.colorbar()

plt.title("Attention Heatmap")

plt.show()


In [ ]:
# -------------------------------------------------------------
# PART 3: Tasks 3.2 & 3.3 Positional and Masking Ablations
# -------------------------------------------------------------
print("--- Executing Positional Encoding Ablation ---")

# Revert global variables back to subword poems
train_data = np.memmap("poems_train.bin", dtype=np.uint16, mode="r")
val_data = np.memmap("poems_val.bin", dtype=np.uint16, mode="r")

no_pos_config = TransformerConfig(
    vocabulary_size=50257,
    context_window=SETTINGS["context_window"],
    num_layers=4,
    num_heads=4,
    embedding_dim=256,
    use_positional_embeddings=False # Completely disables coordinate injections
)

no_pos_model = NanoTransformer(no_pos_config).to(DEVICE)
no_pos_history = train_model(no_pos_model, dataset_name="no_positional", training_steps=300, evaluation_interval=100)

print("\n--- Testing Attention Masking Deletion ---")
# Duplicate your original config but flip off future masking constraints
unmasked_config = TransformerConfig(
    vocabulary_size=50257,
    context_window=SETTINGS["context_window"],
    num_layers=4,
    num_heads=4,
    embedding_dim=256,
    use_causal_mask=False # Disables upper-triangular masking bounds
)

unmasked_model = NanoTransformer(unmasked_config).to(DEVICE)
# Load trained weights into the unmasked model skeleton to see how it performs
unmasked_model.load_state_dict(model.state_dict(), strict=False)

prompt = "The stars in the sky"
input_ids = torch.tensor([TOKENIZER.encode(prompt)], device=DEVICE)

# Run standard generation with the unmasked model
out_unmasked = unmasked_model.generate(input_ids, max_tokens=30, temperature=0.7)
print("Generations with Masking Disabled (Expect structural repetition/garbage):")
print(TOKENIZER.decode(out_unmasked[0].tolist()))

In [ ]:
# -------------------------------------------------------------
# TASK 3.4: Attention Head Dimensionality Experiment
# -------------------------------------------------------------
print("--- Running Attention Head Expressivity Experiment ---")

# Restore global poem data pointers
train_data = np.memmap("poems_train.bin", dtype=np.uint16, mode="r")
val_data = np.memmap("poems_val.bin", dtype=np.uint16, mode="r")

# Configuration A: Single Large Attention Head (dim = 256)
single_head_config = TransformerConfig(
    vocabulary_size=50257,
    context_window=SETTINGS["context_window"],
    num_layers=2,
    num_heads=1,
    embedding_dim=256
)
single_head_model = NanoTransformer(single_head_config).to(DEVICE)

print("\n> Training Single-Head Configuration...")
single_head_history = train_model(single_head_model, dataset_name="single_head", training_steps=200, evaluation_interval=100)

# Plot performance vs the multi-head tracking run to show stability differences
plt.figure(figsize=(8, 4))
plt.plot(single_head_history["step"], single_head_history["validation_loss"], label="Single Large Head (h=1)", color="purple")
plt.plot(poem_history["step"][:3], poem_history["validation_loss"][:3], label="Multi-Head Parallel (h=4)", color="blue")
plt.xlabel("Steps")
plt.ylabel("Validation Loss")
plt.title("Task 3.4: Attention Head Expressivity Optimization Path")
plt.legend()
plt.grid(True)
plt.show()